# Lecture 2 — Practical: *Building and Evaluating Your First Baseline*
### Practical Machine Learning for Transcriptomics in Cancer Research

This is the first **modelling** practical. You take the clean, split METABRIC data from Lecture 1
and build a complete, *trustworthy* baseline:

1. preprocess **inside a pipeline** (fit on train only),
2. train a **regularised Logistic Regression**,
3. evaluate with **imbalance-aware metrics** (precision, recall, F1, ROC-AUC, PR-AUC),
4. **cross-validate honestly** (the whole pipeline inside every fold),
5. diagnose **overfitting** (train vs test, and a regularisation sweep),
6. **interpret** coefficients as biological *hypotheses*,
7. write a short **baseline model report** — the assessable deliverable.

> **The model is the easy part.** As in the lecture, the classifier takes a few lines; the value is
> in doing the pipeline, the evaluation, and the overfitting check *trustworthily*. Adopt the mindset
> of an analyst writing the "baseline model" section of a methods paper — building a defensible
> yardstick, not chasing a number.

#### Continuity & reminders
- **Label:** binary **recurrence** (relapse within the horizon) — *not* pCR. A deliberate
  simplification of time-to-event data (revisited in Section 5 of the course).
- **Class imbalance:** recurrence is the minority class → accuracy will mislead; stratify everything.
- **Leakage discipline (Lecture 1):** every data-dependent step is fit on **training data only**.

> **Network note.** This notebook reuses Lecture 1's real-data loaders (cBioPortal + GEO). If the
> prepared Lecture 1 artefacts are present in the shared `datasets/` cache they are loaded directly;
> otherwise they are regenerated (needs internet). Downloads are git-ignored.


## Section 0 — Setup & framing  *(≈10 min)*

In [ ]:
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import GEOparse  # noqa
except ImportError:
    _pip("GEOparse")

import os, tarfile, io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, accuracy_score)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

def _resolve_data_dir():
    """Find the shared lesson-01 datasets/ cache regardless of where Jupyter launched,
    so this Lecture-2 notebook reuses the data Lecture 1 already downloaded."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        lessons = parent / "lessons"
        if lessons.is_dir():
            for cand in sorted(lessons.glob("lesson01_*/practical/task/datasets")):
                cand.mkdir(parents=True, exist_ok=True); return cand
            for cand in sorted(lessons.glob("lesson01_*/practical/task")):
                d = cand / "datasets"; d.mkdir(parents=True, exist_ok=True); return d
    for c in [here.parent / "datasets", here / "datasets"]:
        if c.parent.exists():
            c.mkdir(parents=True, exist_ok=True); return c
    fb = here / "datasets"; fb.mkdir(parents=True, exist_ok=True); return fb

DATA_DIR = str(_resolve_data_dir())
print("Setup complete. Shared data cache:")
print("  ", os.path.abspath(DATA_DIR))

### Loading the prepared Lecture 1 cohort (shared infrastructure)

The cell below reuses Lecture 1's real loaders and rebuilds the prepared starting point: the
HR+/HER2− expression matrix, the binary recurrence label, and a **patient-level, stratified**
train/validation/test split. Read it, but you don't need to edit it — today's teaching happens
*after* this point. (If you saved Lecture 1's artefacts, you could load them instead; we regenerate
here so the notebook is self-contained.)


In [ ]:
CBIO_URL = "https://cbioportal-datahub.s3.amazonaws.com/brca_metabric.tar.gz"

def _read_cbio_clinical(fileobj):
    return pd.read_csv(fileobj, sep="\t", comment="#", low_memory=False)

def load_metabric(data_dir=DATA_DIR):
    tar_path = os.path.join(data_dir, "brca_metabric.tar.gz")
    if not os.path.exists(tar_path):
        print("Downloading METABRIC from cBioPortal (~50 MB, one time)...")
        r = requests.get(CBIO_URL, stream=True, timeout=120); r.raise_for_status()
        with open(tar_path, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
    with tarfile.open(tar_path, "r:gz") as tf:
        expr = pd.read_csv(tf.extractfile("brca_metabric/data_mrna_illumina_microarray.txt"),
                           sep="\t", low_memory=False)
        pat = _read_cbio_clinical(tf.extractfile("brca_metabric/data_clinical_patient.txt"))
        smp = _read_cbio_clinical(tf.extractfile("brca_metabric/data_clinical_sample.txt"))
    expr = expr.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in expr.columns])
    expr = expr.dropna(subset=["Hugo_Symbol"]).set_index("Hugo_Symbol")
    expr = expr[~expr.index.duplicated(keep="first")]
    clin = pat.merge(smp, on="PATIENT_ID", how="inner", suffixes=("", "_smp"))
    if "SAMPLE_ID" in clin.columns:
        clin = clin.set_index("SAMPLE_ID")
    return expr, clin

def prepare_cohort():
    """Reproduce the Lecture 1 prepared cohort: HR+/HER2- , binary recurrence label,
    samples x genes matrix aligned to the label."""
    expr, clin = load_metabric()
    X = expr.T.copy(); X.index.name = "SAMPLE_ID"
    assert X.shape[1] > X.shape[0] and "ESR1" in X.columns, "orientation check failed"
    common = sorted(set(X.index) & set(clin.index))
    X, clin = X.loc[common], clin.loc[common]
    # HR+/HER2-
    hrpos = clin.get("ER_STATUS").eq("Positive") | clin.get("PR_STATUS").eq("Positive")
    her2neg = clin.get("HER2_STATUS").eq("Negative")
    mask = (hrpos & her2neg).fillna(False)
    X, clin = X.loc[mask], clin.loc[mask]
    # binary recurrence label with explicit censoring policy (horizon = 60 months)
    HORIZON = 60
    status_col = next((c for c in ["RFS_STATUS", "DFS_STATUS"] if c in clin.columns), None)
    months_col = next((c for c in ["RFS_MONTHS", "DFS_MONTHS"] if c in clin.columns), None)
    recurred = clin[status_col].astype(str).str.startswith("1")
    months = pd.to_numeric(clin[months_col], errors="coerce")
    y = pd.Series(index=clin.index, dtype="float")
    y[(recurred) & (months <= HORIZON)] = 1
    y[(~recurred) & (months >= HORIZON)] = 0
    y[(recurred) & (months > HORIZON)] = 0
    keep = y.notna()
    X, clin, y = X.loc[keep], clin.loc[keep], y[keep].astype(int)
    # light gene filter to keep the practical fast & memory-friendly: top-variance genes.
    # (Done on the FULL cohort here only to fix a common feature space BEFORE splitting; it uses
    #  no labels, so it is not outcome leakage. All label-dependent steps stay inside the split.)
    X = X.apply(pd.to_numeric, errors="coerce")
    top_var = X.var(axis=0).sort_values(ascending=False).head(2000).index
    X = X[top_var]
    return X, clin, y

# Build prepared cohort and a patient-level, stratified train/val/test split (60/20/20)
X_all, clin_all, y_all = prepare_cohort()
idx = y_all.index.to_numpy()
tr, tmp = train_test_split(idx, test_size=0.40, random_state=RANDOM_STATE, stratify=y_all.loc[idx])
va, te  = train_test_split(tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_all.loc[tmp])
X_train, y_train = X_all.loc[tr], y_all.loc[tr]
X_val,   y_val   = X_all.loc[va], y_all.loc[va]
X_test,  y_test  = X_all.loc[te], y_all.loc[te]
PATIENT = clin_all["PATIENT_ID"] if "PATIENT_ID" in clin_all.columns else pd.Series(idx, index=idx)

print(f"prepared cohort: {X_all.shape[0]} patients x {X_all.shape[1]} genes")
print(f"split: train {len(tr)} | val {len(va)} | test {len(te)}")
print(f"recurrence prevalence (overall): {y_all.mean():.1%}")

---
## Section 1 — Load and confirm the starting point  *(≈15 min)*

Before modelling, re-verify the split you inherited. Never trust saved artefacts blindly — a silent
misalignment between X and y, or a patient straddling train/test, would invalidate everything below.


> **Exercise 1.1 — re-verify the integrity checks from Lecture 1.**
>
> Confirm: (i) X and y are aligned (same index, same order) in each split; (ii) **no patient appears
> in two splits**; (iii) the recurrence proportion is **preserved** across train/val/test (stratification
> held). Print a small table of n and positive-rate per split.
>
> *Hint:* `PATIENT.loc[<index>]` gives patient IDs; use set intersections for overlap.


In [ ]:
# TODO 1.1
# (i) assert X and y share index & order in each split.
# (ii) assert ZERO patient overlap across train/val/test (use PATIENT.loc[...] and set ops).
# (iii) build a small table of n / positives / positive_rate per split; check stratification held.
#
# for name, Xs, ys in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
#     assert ...
# pt = {...}
# tbl = pd.DataFrame({...})


> **Discussion 1.** Why begin a *modelling* notebook by re-verifying the split, rather than trusting
> the saved files? (What silent failure would otherwise go undetected — and invalidate every number
> you compute afterwards?)


---
## Section 2 — Preprocessing inside a pipeline  *(≈30 min)*

We assemble preprocessing + model into a single `Pipeline` so that **all** data-dependent steps are
fit on training data only and applied to validation/test — leakage becomes the path of *most*
resistance.


> **Exercise 2.1 — build the pipeline; confirm fit-on-train-only.**
>
> Assemble a `Pipeline` of: median imputation → standardisation → (we'll add the model in Section 3).
> Fit it on **train**, transform val/test. Then *prove* the scaler used only training statistics — e.g.
> show the scaler's learned mean equals the **training** column means, not the whole-cohort means.
>
> *Hint:* after `pipe.fit(X_train)`, the scaler step exposes `.mean_`; compare to `X_train.mean()`.


In [ ]:
# TODO 2.1
# Build a Pipeline: SimpleImputer(median) -> StandardScaler. Fit on X_train only; transform val.
# Then prove the scaler used train statistics: compare scaler.mean_ to training column means
# (after imputation) vs whole-cohort means.
#
# preprocess = Pipeline([("impute", ...), ("scale", ...)])
# preprocess.fit(X_train)
# learned_mean = preprocess.named_steps["scale"].mean_
# ... compare to train vs whole-cohort means ...


> **Exercise 2.2 — the leakage trap (quantify the optimism).**
>
> Do the **wrong** thing on purpose: fit the scaler on the **whole** matrix *before* splitting, train a
> quick model, and read a validation score. Then do it **correctly** (scale fit on train only) and read
> the validation score again. Report the gap — the *optimism* leakage buys you.
>
> *Hint:* use a simple regularised `LogisticRegression` for both; only the scaling step differs.


In [ ]:
# TODO 2.2  (leakage trap)
# WRONG path: fit a StandardScaler on the WHOLE matrix, then split, then train -> read val ROC-AUC.
# RIGHT path: scaler fit on train only -> read val ROC-AUC.
# Print both and the optimism gap (leaky minus honest).
#
# quick_model = lambda: LogisticRegression(penalty="l2", C=0.1, max_iter=2000, random_state=RANDOM_STATE)
# ... leaky ... ; ... honest ... ; print gap


> **Discussion 2.** Why does the leaky version look *better*? What does that teach you about trusting
> a reported number when you don't know whether preprocessing happened before or after the split?
>
> *Misconception to retire:* that scaling is harmless "data cleaning" outside the model.


---
## Section 3 — Train a regularised Logistic Regression  *(≈30 min)*

Now add the model. We compare a barely-regularised model with a well-regularised one, and look at
sparsity. We read **training** performance only — and explicitly withhold judgement until validation.


> **Exercise 3.1 — weak vs strong regularisation (training only).**
>
> Build two full pipelines (impute → scale → Logistic Regression): one **weakly** regularised (large
> `C`), one **well** regularised (small `C`). Fit on train; report **training** ROC-AUC for each. Note
> which looks better *on training* — and resist drawing conclusions yet.
>
> *Hint:* in scikit-learn, larger `C` = weaker regularisation; smaller `C` = stronger.


In [ ]:
# TODO 3.1
# Write make_pipe(C, ...) returning impute->scale->LogisticRegression.
# Fit a weakly regularised (large C) and a well-regularised (small C) pipeline on train.
# Print TRAINING ROC-AUC for each. Note which looks better on training. Do NOT peek at validation.
#
# def make_pipe(C, ...): ...
# weak = make_pipe(C=100.0).fit(X_train, y_train); strong = make_pipe(C=0.05).fit(...)


> **Exercise 3.2 — sparsity: how many genes survive?**
>
> Fit a **LASSO-style** Logistic Regression (`penalty="l1"`, `solver="liblinear"` or `"saga"`) and count
> how many genes have **non-zero** coefficients, versus the dense (`l2`) model. The L1 model selects a
> small panel automatically.
>
> *Hint:* `pipe.named_steps["clf"].coef_` is the coefficient vector.


In [ ]:
# TODO 3.2
# Fit a LASSO-style pipeline (penalty="l1", solver="liblinear") and a dense (l2) one.
# Count non-zero coefficients in each (pipe.named_steps["clf"].coef_). Report the contrast.
#
# lasso = make_pipe(C=0.1, penalty="l1", solver="liblinear").fit(X_train, y_train)
# n_lasso = int(np.sum(lasso.named_steps["clf"].coef_[0] != 0)) ...


> **Discussion 3.** Before we validate — which model do you *expect* to generalise better, the weak or
> the strong one, and why? Write your prediction; we test it next.


---
## Section 4 — Evaluate with the right metrics  *(≈35 min)*

Now we look at validation — with metrics that respect the class imbalance. Accuracy alone will lie to
us; we use the confusion matrix, precision/recall/F1, ROC-AUC **and** PR-AUC.


> **Exercise 4.1 — the full metrics table, and the trivial-classifier contrast.**
>
> Pick the **well-regularised** baseline (`strong` from 3.1). On the **validation** split, compute:
> accuracy, precision, recall, F1, ROC-AUC, PR-AUC. Then compute the accuracy of a **trivial "always
> predict no-recurrence"** classifier and contrast it with your model's accuracy.
>
> *Hint:* PR-AUC = `average_precision_score`; the trivial accuracy = 1 − prevalence.


In [ ]:
# TODO 4.1
# Using the well-regularised baseline, predict on VALIDATION. Compute accuracy, precision, recall,
# F1, ROC-AUC, PR-AUC (average_precision_score). Then compute the trivial 'always negative' accuracy
# (= 1 - prevalence) and contrast.
#
# proba_val = baseline.predict_proba(X_val)[:,1]; pred_val = (proba_val>=0.5).astype(int)
# metrics = {...}


> **Exercise 4.2 — move the threshold; trade precision against recall.**
>
> Sweep the decision threshold (e.g. 0.2 → 0.8) and plot precision and recall versus threshold. Then
> **choose** an operating threshold justified by the clinical cost asymmetry (missing a recurrence is
> usually worse than a false alarm). State your choice in a comment.


In [ ]:
# TODO 4.2
# Sweep thresholds 0.1..0.9; compute precision and recall at each; plot both vs threshold.
# Choose an operating threshold justified by clinical cost asymmetry (favour recall here) and
# state it in a comment, with the recall/precision it yields.
#
# ths = np.linspace(0.1, 0.9, 33); precs = [...]; recs = [...]


> **Exercise 4.3 — ROC vs PR (and why ROC flatters).**
>
> Plot the ROC curve and the PR curve for the baseline on validation, annotating ROC-AUC and PR-AUC.
> Mark the chance line on ROC and the **prevalence** line on PR. Then explain in writing why ROC looks
> more flattering than PR here.


In [ ]:
# TODO 4.3
# Plot ROC (with chance line) and PR (with prevalence line), annotate ROC-AUC and PR-AUC.
# Then write 2-3 sentences: why does ROC look more flattering than PR under class imbalance?
#
# fpr, tpr, _ = roc_curve(y_val, proba_val); prec_c, rec_c, _ = precision_recall_curve(y_val, proba_val)


> **Discussion 4.** Which single metric would you put in an abstract for this problem — and which would
> a careful reviewer *also* demand to see? *Misconception to retire:* that a high ROC-AUC implies the
> model is clinically useful.


---
## Section 5 — Cross-validation, done honestly  *(≈35 min)*

A single validation split is a noisy ruler. We cross-validate with the **entire pipeline inside every
fold**, then expose the most common silent error in omics ML: a *leaky* CV that looks better and is
wrong.


> **Exercise 5.1 — honest stratified CV vs the single split.**
>
> Run **stratified k-fold** cross-validation of the well-regularised pipeline on the *training* data
> (the whole pipeline refits inside each fold — `cross_val_score` does this for a `Pipeline`). Report
> the mean ± SD ROC-AUC, and compare to your single-split validation AUC from Section 4.
>
> *Hint:* `cross_val_score(pipe, X_train, y_train, cv=StratifiedKFold(5, shuffle=True), scoring="roc_auc")`.


In [ ]:
# TODO 5.1
# Run stratified 5-fold CV of the well-regularised pipeline on X_train/y_train (scoring="roc_auc").
# Report per-fold scores and mean ± SD. Compare to the single-split validation AUC from Section 4.
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# cv_scores = cross_val_score(make_pipe(C=0.05), X_train, y_train, cv=cv, scoring="roc_auc")


> **Exercise 5.2 — the leakage hunt (the payoff).**
>
> Build a **leaky** cross-validation: select the top-k genes by association with the outcome using
> **all** the training data *once*, then cross-validate on those fixed genes. Compare its score to an
> **honest** CV where gene selection happens **inside each fold**. Identify, in writing, the exact step
> that leaks.
>
> *Hint:* honest = put `SelectKBest(f_classif, k=...)` *inside* the `Pipeline` so it refits per fold;
> leaky = run `SelectKBest` on all of `X_train` first, then CV on the reduced matrix.


In [ ]:
# TODO 5.2  (leakage hunt — the payoff)
# LEAKY: run SelectKBest(f_classif, k=100) on ALL of X_train once, then CV a scale->LR pipeline
#        on the reduced matrix.
# HONEST: put SelectKBest INSIDE the pipeline (impute->select->scale->clf) so it refits per fold; CV that.
# Print both CV means and the optimism gap; write one sentence naming the exact leaking step.
#
# K = 100
# leaky: sel_all = SelectKBest(...).fit(impute(X_train), y_train); X_train_reduced = ...
# honest: Pipeline([("impute",...),("select",SelectKBest(...)),("scale",...),("clf",...)])


> **Discussion 5.** Why can a *leaky* cross-validation be **more** dangerous than a single bad split?
> (Hint: it produces a confident, stable-looking number — it *launders* the leakage.)


---
## Section 6 — Train vs test, and watching overfitting  *(≈25 min)*

We make overfitting visible: first the train-vs-test gap of an under-regularised model, then the
regularisation sweep that reveals the validation sweet spot.


> **Exercise 6.1 — the overfitting gap.**
>
> Fit a deliberately **under-regularised** model (large `C`). Compare its **training** ROC-AUC to its
> **cross-validated** ROC-AUC. Plot the two as bars and annotate the gap — the fingerprint of overfitting.


In [ ]:
# TODO 6.1
# Fit an under-regularised pipeline (large C, e.g. 1000). Compute training ROC-AUC and CV ROC-AUC.
# Plot as two bars, annotate values, and report the gap.
#
# overfit_pipe = make_pipe(C=1000.0).fit(X_train, y_train)
# auc_train_of = ...; auc_cv_of = cross_val_score(..., cv=cv, scoring="roc_auc").mean()


> **Exercise 6.2 — the regularisation sweep.**
>
> Sweep `C` across several orders of magnitude. For each, record **training** ROC-AUC and
> **cross-validated** ROC-AUC. Plot both versus `C` (log x-axis). Identify the validation **peak** (the
> sweet spot) and the over-complex region where validation falls while training keeps rising.


In [ ]:
# TODO 6.2
# Sweep C over np.logspace(-3, 3, 13). For each C record training ROC-AUC and CV ROC-AUC.
# Plot both vs C on a log x-axis; mark the validation peak (sweet spot).
#
# Cs = np.logspace(-3, 3, 13); train_aucs=[]; cv_aucs=[]
# for C in Cs: ...


> **Discussion 6.** Your training AUC is ~0.99 and your CV AUC is ~0.65. What is the model telling you,
> and what do you change? *Misconception:* that more training performance is always progress.


---
## Section 7 — Interpret the coefficients (carefully)  *(≈20 min)*

Coefficients rank genes by contribution — a starting point for biology, and the reason we like linear
baselines. But predictive ≠ causal, and selected coefficients are unstable. Read them as hypotheses.


> **Exercise 7.1 — top genes, with a plausibility check.**
>
> Refit the well-regularised baseline on the training data; extract the largest **positive** and
> **negative** coefficients with their gene symbols; plot them. Comment on biological plausibility
> (proliferation markers → recurrence; ER-signalling/luminal genes → no recurrence).
>
> *Hint:* `coef = pipe.named_steps["clf"].coef_[0]`; map to `X_train.columns`.


In [ ]:
# TODO 7.1
# Refit the well-regularised baseline; extract coef_ as a Series indexed by X_train.columns.
# Take top +ve and top -ve coefficients; plot a horizontal bar chart with gene labels.
# Comment on biological plausibility.
#
# base = make_pipe(C=0.05).fit(X_train, y_train)
# coef = pd.Series(base.named_steps["clf"].coef_[0], index=X_train.columns)


> **Exercise 7.2 — stability across resamples (ties to Lecture 1).**
>
> Refit the baseline on several bootstrap resamples (or CV folds) of the training data; record which
> genes land in the top-k each time, and report how *often* each top gene reappears. Unstable
> membership is the p ≫ n "signatures are not unique" lesson, live.


In [ ]:
# TODO 7.2
# For N bootstrap resamples of the training data: refit the baseline, record the top-K genes by |coef|.
# Count how often each gene appears; report the most stable genes (fraction of resamples).
# Relate to Lecture 1's 'signatures are not unique'.
#
# from collections import Counter; counter = Counter(); rng = np.random.default_rng(RANDOM_STATE)
# for _ in range(20): bs = rng.choice(X_train.index, size=len(X_train), replace=True); ...


> **Discussion 7.** If your top gene is a known proliferation marker, have you discovered biology — or
> just rediscovered tumour grade (a cheap clinical variable)? (This previews Lecture 3.) *Misconception:*
> that coefficient magnitude equals biological importance.


---
## Section 8 — Baseline model report  *(≈15 min — the assessable deliverable)*

Write a **200–300 word "baseline model report"**: what the baseline achieves (with the right metric and
its uncertainty), how it was validated, where it might be overfitting, what the coefficients suggest as
*hypotheses*, and what you would need before trusting it. This is the Section-2 course deliverable.


**Baseline model report (200–300 words):**

*(TODO — write your report here. Cover: headline metric + uncertainty from CV; the right metric under
imbalance and your chosen threshold; the train-vs-CV overfitting gap and the regularisation you chose;
the top coefficient hypotheses and their stability; and what you'd need before trusting this as a
recurrence biomarker.)*


---
### Deliverables checklist
- [ ] Verified, aligned train/val/test artefacts (Section 1)
- [ ] Preprocessing pipeline + quantified leakage effect (Section 2)
- [ ] Two fitted baselines + selected-gene count (Section 3)
- [ ] Metrics table + ROC and PR plots + justified threshold (Section 4)
- [ ] Honest CV score with spread + honest-vs-leaky comparison (Section 5)
- [ ] Overfitting-gap figure + regularisation sweep with chosen strength (Section 6)
- [ ] Ranked coefficients + stability check + predictive-vs-causal note (Section 7)
- [ ] Final baseline model report (Section 8)

> **The message of the lecture, in one line:** *a well-validated baseline model is more valuable than a
> sophisticated model with poor methodology.* Everything you did here is that rigour — and you can now
> build, evaluate, **and critique** a baseline.
